# 02 — Real 5G data acquisition and schema inspection
Run all cells in order. **CPU runtime is sufficient.** This self-contained notebook needs no updates to your previous checkout.

It inventories the public Korean 5G dataset and downloads up to three small CSV archives (maximum 64 MiB each), then reads the first 5,000 rows from each. Samples are for schema inspection, not training or representative sampling. Raw data stays in your Drive and is not pushed to GitHub.

The provider currently reports the license as Unknown. We record that uncertainty; downloading a public sample does not establish permission to redistribute it or approve it as publication data. No credentials are requested. If access requires authentication, the notebook records the failure; do not paste tokens into chat.

Source: https://www.kaggle.com/datasets/kimdaegyeom/5g-traffic-datasets


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import datetime, json
PROJECT = Path('/content/drive/MyDrive/5G_QoS_Research')
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = PROJECT / 'data' / 'real_inspection' / STAMP
OUT.mkdir(parents=True, exist_ok=False)
print('Outputs:', OUT)


In [ ]:
"""Bounded acquisition of public Kaggle files for schema inspection."""
import hashlib, io, json, time, urllib.parse, urllib.request, zipfile
from pathlib import Path
import pandas as pd
BASE = 'https://www.kaggle.com/api/v1/datasets/'
DATASET = 'kimdaegyeom/5g-traffic-datasets'

def get_json(url):
    with urllib.request.urlopen(url, timeout=60) as response:
        raw = response.read(8 * 1024 * 1024 + 1)
    if len(raw) > 8 * 1024 * 1024:
        raise ValueError('Metadata response exceeded limit')
    return json.loads(raw)

def inventory():
    metadata = get_json(BASE + 'view/' + DATASET)
    files, tokens = {}, set()
    token = None
    for _ in range(100):
        url = BASE + 'list/' + DATASET
        if token:
            url += '?' + urllib.parse.urlencode({'pageToken': token})
        page = get_json(url)
        for item in page.get('datasetFiles', []):
            name = item.get('name') or item.get('nameNullable')
            if name:
                files[name] = {'name': name, 'bytes': int(item.get('totalBytes', 0))}
        token = page.get('nextPageToken') or page.get('nextPageTokenNullable')
        if not token:
            return metadata, list(files.values())
        if token in tokens:
            raise RuntimeError('Repeated pagination token; refusing incomplete inventory')
        tokens.add(token)
    raise RuntimeError('Inventory page limit exceeded')

def acquire_preview(item, destination, rows=5000, max_download=64*1024*1024):
    """Download one bounded archive and parse only a prefix. No raw IPs printed."""
    name = item['name']
    if not name.endswith('.csv'):
        raise ValueError('Only CSV files supported')
    url = BASE + 'download/' + DATASET + '/' + urllib.parse.quote(name, safe='') + '?datasetVersionNumber=1'
    directory = Path(destination)
    directory.mkdir(parents=True, exist_ok=True)
    digest = hashlib.sha256(name.encode()).hexdigest()[:12]
    archive = directory / (digest + '.download')
    temp = archive.with_suffix('.partial')
    try:
        with urllib.request.urlopen(url, timeout=60) as response, temp.open('wb') as target:
            size = 0
            while True:
                block = response.read(min(1024*1024, max_download-size+1))
                if not block: break
                size += len(block)
                if size > max_download:
                    raise ValueError('File exceeds 64 MiB download cap; choose a smaller source file')
                target.write(block)
        temp.replace(archive)
    finally:
        temp.unlink(missing_ok=True)
    sha = hashlib.sha256(archive.read_bytes()).hexdigest()
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as z:
            members = [m for m in z.infolist() if not m.is_dir() and m.filename.endswith('.csv')]
            if len(members) != 1:
                raise ValueError('Expected single CSV in file download')
            member = members[0]
            if member.file_size > 2*1024**3:
                raise ValueError('Uncompressed file exceeds inspection limit')
            with z.open(member) as stream:
                df = pd.read_csv(stream, nrows=rows)
    else:
        df = pd.read_csv(archive, nrows=rows)
    sample = directory / (digest + '_prefix.csv')
    df.to_csv(sample, index=False)
    parts = Path(name).parts
    temporal = {}
    if 'Time' in df:
        parsed = pd.to_datetime(df['Time'], errors='coerce')
        temporal = {'parse_failures': int(parsed.isna().sum()),
                    'monotonic_in_file': bool(parsed.is_monotonic_increasing),
                    'prefix_span_seconds': float((parsed.max()-parsed.min()).total_seconds()) if parsed.notna().any() else None,
                    'timezone': 'not established; no cross-capture chronology assumed'}
    record = {
        'timestamp_inspection': temporal,
        'dataset': DATASET, 'requested_version':1, 'source_file':name,
        'source_declared_bytes':item['bytes'], 'download_bytes':archive.stat().st_size,
        'download_sha256':sha, 'prefix_csv_sha256':hashlib.sha256(sample.read_bytes()).hexdigest(),
        'rows_inspected':len(df), 'columns':list(df.columns),
        'dtypes':{c:str(df[c].dtype) for c in df},
        'missing':{c:int(df[c].isna().sum()) for c in df},
        'unique_counts':{c:int(df[c].nunique()) for c in df},
        'category_from_path':parts[-3] if len(parts)>=3 else None,
        'application_from_path':parts[-2] if len(parts)>=2 else None,
        'label_status':'capture-level path labels; not independently validated per packet',
        'sampling':'first rows only; schema inspection, not representative benchmark',
        'training_ready':False,
        'pending':['license verification','timestamp semantics','packet ordering','flow/session independence','label validation'],
    }
    (directory/(digest+'_report.json')).write_text(json.dumps(record,indent=2))
    return record


In [ ]:
metadata, files = inventory()
(OUT / 'provider_metadata.json').write_text(json.dumps(metadata, indent=2))
(OUT / 'file_inventory.json').write_text(json.dumps(files, indent=2))
file_table = pd.DataFrame(files)
file_table.to_csv(OUT / 'file_inventory.csv', index=False)
print('Files:', len(files))
print('Declared total GB:', round(file_table['bytes'].sum()/1e9, 2))
print('Provider license:', metadata.get('licenseName', metadata.get('licenseNameNullable')))
print('Provider version:', metadata.get('currentVersionNumber'))
# Metadata changes must be reviewed rather than silently changing the experiment.
if metadata.get('currentVersionNumber', metadata.get('currentVersionNumberNullable')) != 1:
    raise RuntimeError('Provider version changed. Share metadata summary before acquisition.')
from IPython.display import display
display(file_table.sort_values('bytes').head(15))


In [ ]:
# Select up to three distinct category folders, preferring smaller captures.
# Selection is for schema inspection only, not a balanced training sample.
selected, categories = [], set()
for item in sorted(files, key=lambda x: (x['bytes'], x['name'])):
    parts = Path(item['name']).parts
    if not item['name'].endswith('.csv') or item['bytes'] <= 0 or item['bytes'] > 200_000_000:
        continue
    category = parts[-3] if len(parts)>=3 else 'unknown'
    if category not in categories:
        selected.append(item)
        categories.add(category)
    if len(selected) == 3: break
if not selected:
    raise RuntimeError('No bounded candidates found; share inventory summary.')
(OUT / 'selected_files.json').write_text(json.dumps(selected, indent=2))
print(json.dumps(selected, indent=2))


In [ ]:
reports, errors = [], []
for item in selected:
    print('Inspecting:', item['name'])
    try:
        reports.append(acquire_preview(item, OUT / 'samples'))
    except Exception as error:
        errors.append({'file': item['name'], 'error_type': type(error).__name__, 'message': str(error)[:300]})
summary = {
    'stage':'real_data_schema_inspection',
    'license':metadata.get('licenseName', metadata.get('licenseNameNullable')),
    'files_in_inventory':len(files), 'successful_previews':len(reports),
    'reports':reports, 'errors':errors, 'training_ready':False,
    'next_step':'Validate actual columns and capture labels; then define flow grouping and acquisition for independent train/validation/test captures.'
}
(OUT / 'inspection_summary.json').write_text(json.dumps(summary, indent=2))
print('=== COPY THE JSON BELOW ===')
print(json.dumps(summary, indent=2))
print('Saved summary:', OUT / 'inspection_summary.json')


## What to send back
Download **inspection_summary.json** from the printed Drive folder and attach it in chat. This is easier than copying long output. It contains column names and counts, not raw packet rows. Do not share raw captures or credentials.

No models are trained in this notebook. A successful preview does not establish sufficient independent captures for evaluation. All six families remain planned; IP embeddings require legitimate identifiers and recurrent models require validated packet histories.
